# 🧹 Limpieza de Datos
**Objetivo:** Procesar y estandarizar los datasets finales del proyecto (`dataset_devto_pure_labels_clean_en.csv` /
`dataset_devto_pure_labels_clean_es.csv`) para dejarlos listos para NLP.

**Como correr este notebook para los 2 idiomas:** el notebook procesa **un idioma por corrida**. Corre todo el
notebook (`Run All`) con `IDIOMA = "en"`, y al final vas a tener `dataset_limpio_en.csv`. Despues cambia la celda de
configuracion a `IDIOMA = "es"` y corre todo el notebook otra vez (`Run All`) para generar `dataset_limpio_es.csv`.
No hace falta cargar los dos archivos al mismo tiempo ni mezclarlos: son dos pasadas independientes del mismo
pipeline, una por idioma.

### ✅ Tareas cubiertas en este Notebook:
- [x] Seleccionar idioma a procesar (EN o ES) y cargar el dataset correspondiente
- [x] Mostrar distribución de categorías (value_counts)
- [x] Detectar valores nulos o vacíos
- [x] Función de limpieza: lowercase, sin acentos, sin puntuación, sin números
- [x] Guardar texto limpio en nueva columna
- [x] Combinar titulo + texto para enriquecer representación
- [x] Instalación de librerías y configuración del entorno (100% compatible con Google Colab).
- [x] Carga de datos (separador `,`, UTF-8 con BOM).
- [x] Exploración inicial, matriz de nulos y verificación de tipos.
- [x] Limpieza estructural: Eliminación de duplicados, registros vacíos implícitos (solo espacios) y manejo de valores faltantes.
- [x] Limpieza avanzada de texto: Eliminación de etiquetas HTML (Adición), caracteres especiales, normalización de espacios.
- [x] Validación estructurada de URLs mediante Regex.
- [x] Feature Engineering: Estadísticas de longitud y filtro de anomalías (textos muy cortos) (Adición).
- [x] Gráficos de distribución (variables continuas y categóricas).
- [x] Exportación automática a `CSV` y `Parquet`, con sufijo de idioma en el nombre de archivo.


### 1. Instalación de Dependencias
Aseguramos que Google Colab o el entorno local tengan todas las librerías necesarias.


In [ ]:
# Instalación automática de librerías faltantes

!pip install -q missingno unidecode pyarrow fastparquet beautifulsoup4


### 2. Importación de Librerías y Configuración
Organizamos las importaciones y configuramos las opciones de visualización de `pandas` y gráficos.


In [ ]:
# Importaciones organizadas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import re
import warnings
from unidecode import unidecode
from bs4 import BeautifulSoup

# Configuración de pandas para mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Configuración de gráficos
plt.style.use('ggplot')
warnings.filterwarnings('ignore')

print("✅ Librerías importadas y entorno configurado.")


### 2.1 Seleccion de idioma

Este notebook procesa un idioma a la vez. Cambia `IDIOMA` a `"en"` o `"es"` y ejecuta todo el notebook (`Run All`)
para generar el `dataset_limpio_<idioma>.csv` correspondiente. Para tener ambos idiomas, se corre el notebook dos
veces (una por cada valor de `IDIOMA`), no se cargan los dos archivos juntos.

In [ ]:
# Selecciona el idioma a procesar: "en" (ingles) o "es" (espanol)
IDIOMA = "en"  # cambiar a "es" para procesar la version en espanol

archivos_entrada = {
    "en": "dataset_devto_pure_labels_clean_en.csv",
    "es": "dataset_devto_pure_labels_clean_es.csv",
}
assert IDIOMA in archivos_entrada, "IDIOMA debe ser 'en' o 'es'"

print(f"✅ Idioma seleccionado: {IDIOMA.upper()}")
print(f"Archivo de entrada: {archivos_entrada[IDIOMA]}")


### 3. Carga del Dataset
Cargamos el archivo CSV respetando la codificación y el separador indicados.


In [ ]:
# Carga del CSV (busca primero en ../data/, si no existe intenta /content/ para Google Colab)
nombre_archivo = archivos_entrada[IDIOMA]

try:
    file_path = f"../data/{nombre_archivo}"
    df = pd.read_csv(file_path, encoding='utf-8-sig')
except FileNotFoundError:
    file_path = f"/content/{nombre_archivo}"
    df = pd.read_csv(file_path, encoding='utf-8-sig')

print(f"✅ Dataset cargado exitosamente desde: {file_path}")
print(f"📊 Dimensiones iniciales: {df.shape[0]} filas y {df.shape[1]} columnas.")


### 4. Exploración Inicial de los Datos
Revisamos las primeras filas, la estructura de tipos de datos y las estadísticas generales.


In [ ]:
# Visualización inicial
display(df.head(3))

# Información del dataset (tipos de datos y nulos preliminares)
print("\n--- ℹ️ Información del Dataset ---")
df.info()

# Estadísticas descriptivas para columnas numéricas y categóricas
print("\n--- 📈 Estadísticas Descriptivas ---")
display(df.describe(include='all'))


### 5. Análisis y Tratamiento de Nulos y Duplicados
Visualizaremos los valores faltantes, eliminaremos duplicados absolutos y limpiaremos registros que están compuestos únicamente por espacios.


In [ ]:
# 5.1 Eliminación de duplicados
duplicados_iniciales = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f"🗑️ Se eliminaron {duplicados_iniciales} filas duplicadas absolutas.")

# 5.2 Limpieza de espacios en blanco y registros vacíos
# Reemplazamos strings que solo contienen espacios por NaN
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 5.3 Conteo de valores nulos
print("\n--- 🔍 Conteo de Valores Nulos ---")
display(df.isnull().sum())

# 5.4 Visualización de nulos (Matriz de missingno)
plt.figure(figsize=(10, 6))
msno.matrix(df, figsize=(12, 6), fontsize=10, color=(0.2, 0.4, 0.6))
plt.title('Matriz de Valores Faltantes', fontsize=16)
plt.show()

# 5.5 Eliminación de registros vacíos críticos
# Si no hay texto o título, el registro no sirve para NLP
df.dropna(subset=['titulo', 'texto'], inplace=True)
print(f"✅ Registros restantes tras limpiar nulos críticos: {df.shape[0]}")


### 6. Limpieza y Normalización de Texto
Limpiamos las columnas de texto, eliminamos HTML oculto (adición de robustez), normalizamos espacios y quitamos caracteres extraños.


In [ ]:
def limpiar_texto(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto)
    # Adición: Limpiar etiquetas HTML residuales
    texto = BeautifulSoup(texto, "html.parser").get_text(separator=" ")
    # Limpiar saltos de línea y tabulaciones
    texto = re.sub(r'[\n\t\r]', ' ', texto)
    # Eliminar caracteres especiales repetitivos o no alfanuméricos (mantenemos puntuación básica)
    texto = re.sub(r'[^a-zA-Z0-9\s.,!?¿¡áéíóúÁÉÍÓÚñÑ-]', ' ', texto)
    # Normalizar múltiples espacios
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# El dataset final (EN o ES) solo trae 'titulo' y 'texto' como columnas de texto libre
columnas_texto = ['titulo', 'texto']

print("🧹 Limpiando columnas de texto...")
for col in columnas_texto:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_texto)
        print(f" - Columna '{col}' procesada.")


### 7. Validación de URLs y Verificación de Categorías
Validamos que las URLs tengan un formato correcto y estandarizamos la categoría para evitar duplicidades por mayúsculas/espacios.


In [ ]:
# 7.1 Validación de URLs con Regex
# Expresión regular robusta para URLs
url_pattern = re.compile(r'^(https?:\/\/)?([\da-z\.-]+)\.([a-z\.]{2,6})([\/\w \.-]*)*\/?$')

if 'url' in df.columns:
    df['url_valida'] = df['url'].apply(lambda x: bool(url_pattern.match(str(x).strip())) if pd.notnull(x) else False)
    invalidas = (~df['url_valida']).sum()
    print(f"🔗 URLs evaluadas. Se detectaron {invalidas} URLs con formato inválido o nulas.")

# 7.2 Verificación de Categorías
if 'categoria' in df.columns:
    # Estandarizar a minúsculas y quitar espacios
    df['categoria'] = df['categoria'].astype(str).str.lower().str.strip()

    print("\n🏷️ Distribución de Categorías:")
    display(df['categoria'].value_counts())


### 8. Limpieza de Keywords

Los datasets finales (`dataset_devto_pure_labels_clean_en/es.csv`) no traen columnas de keywords — esta sección se
deja por compatibilidad con el resto del pipeline del equipo (si en el futuro se agregan columnas `keywords_*`, se
limpian automáticamente aquí; si no existen, el paso simplemente no hace nada).

In [ ]:
def limpiar_keywords(kw):
    if pd.isna(kw):
        return ""
    kw = str(kw).lower()
    # Eliminar corchetes y comillas
    kw = re.sub(r'[\[\]\'\"]', '', kw)
    # Separar por comas, limpiar espacios y rearmar
    palabras = [word.strip() for word in kw.split(',')]
    # Retornar separadas por coma, omitiendo las vacías
    return ", ".join(filter(None, palabras))

columnas_kw = ['keywords_titulo', 'keywords_texto', 'keywords_titulo_esp', 'keywords_texto_esp']

print("🔑 Limpiando columnas de keywords...")
for col in columnas_kw:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_keywords)
        print(f" - Columna '{col}' procesada.")


### 9. Feature Engineering (Adiciones extra de limpieza)
Calculamos las longitudes del texto. Esto nos sirve para visualizar y también para filtrar **registros anómalos** (textos extremadamente cortos que no aportan al NLP).


In [ ]:
# Calcular longitudes
df['longitud_titulo'] = df['titulo'].str.len().fillna(0)
df['longitud_texto'] = df['texto'].str.len().fillna(0)

# Detección de anomalías: Textos muy cortos (< 30 caracteres)
umbral_corto = 30
anomalos = df[df['longitud_texto'] < umbral_corto]
print(f"⚠️ Textos extremadamente cortos detectados (menores a {umbral_corto} caracteres): {len(anomalos)}")

# Filtramos esos textos anómalos para tener un corpus limpio
df = df[df['longitud_texto'] >= umbral_corto]
print(f"✅ Dataset tras filtrar anomalías: {df.shape[0]} registros.")


### 10. Gráficos de Distribución
Visualizamos cómo quedaron nuestras distribuciones tras la limpieza.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de Títulos
sns.histplot(df['longitud_titulo'], bins=40, kde=True, ax=axes[0], color='#1f77b4')
axes[0].set_title('Distribución de Longitud de Títulos', fontsize=13)
axes[0].set_xlabel('Caracteres')

# Histograma de Textos
sns.histplot(df['longitud_texto'], bins=40, kde=True, ax=axes[1], color='#ff7f0e')
axes[1].set_title('Distribución de Longitud de Textos', fontsize=13)
axes[1].set_xlabel('Caracteres')

plt.tight_layout()
plt.show()

# Gráfico de Categorías
if 'categoria' in df.columns:
    plt.figure(figsize=(10, 5))
    conteo = df['categoria'].value_counts()
    sns.barplot(y=conteo.index, x=conteo.values, palette='viridis')
    plt.title('Frecuencia por Categoría', fontsize=14)
    plt.xlabel('Cantidad de Artículos')
    plt.ylabel('Categoría')
    plt.show()


### 11. Exportación de Datos
El dataset está limpio, libre de valores basura, normalizado y listo para modelado (NLP). Exportamos a `CSV` y `Parquet` (este último es más eficiente en lectura/escritura y respeta estrictamente los tipos de datos).


In [ ]:
# (Opcional) Eliminar las columnas auxiliares creadas si no se necesitan en NLP
# df.drop(columns=['url_valida', 'longitud_titulo', 'longitud_texto'], inplace=True)

archivo_csv = f'dataset_limpio_{IDIOMA}.csv'
archivo_parquet = f'dataset_limpio_{IDIOMA}.parquet'

# Exportar a CSV
df.to_csv(archivo_csv, index=False, encoding='utf-8')

# Exportar a Parquet
df.to_parquet(archivo_parquet, index=False)

print("🎉 ¡Proceso Finalizado con Éxito!")
print(f"Idioma procesado: {IDIOMA.upper()}")
print(f"Dimensiones finales del dataset: {df.shape[0]} filas x {df.shape[1]} columnas.")
print(f"Archivos listos para descargar:\n 1. {archivo_csv}\n 2. {archivo_parquet}")

otro_idioma = "es" if IDIOMA == "en" else "en"
print(f"\n➡️ Para generar el archivo del otro idioma: cambia IDIOMA a \"{otro_idioma}\" en la celda de configuracion "
      f"(seccion 2.1) y ejecuta todo el notebook de nuevo (Run All).")
